### Graph Analytics

Represent entities as nodes and relationships as edges, useful specifically when the CONNECTIONS between data points carry signal a flat tabular row cannot capture. Directly relevant to fraud: fraud rings show up as unusually dense clusters or high-centrality nodes in a graph of shared identifiers (phone numbers, devices, payment methods across accounts that look unrelated in a flat table).

#### 0. Representations

Toy graph, 5 accounts, an edge means two accounts share a payment method or device:
```
edges: A-B, A-C, B-C, C-D, D-E
```
(A, B, C form a tight triangle, all pairwise connected; C also bridges to D; D bridges to the otherwise-isolated E)

Adjacency matrix, symmetric for an undirected graph, 1 if an edge exists:
```
     A  B  C  D  E
  A  0  1  1  0  0
  B  1  0  1  0  0
  C  1  1  0  1  0
  D  0  0  1  0  1
  E  0  0  0  1  0
```
Adjacency list, more compact for sparse graphs (most real graphs are sparse, most pairs of nodes are NOT connected): `{A: [B,C], B: [A,C], C: [A,B,D], D: [C,E], E: [D]}`.

In [ ]:
import networkx as nx

G = nx.Graph()
G.add_edges_from([("A","B"), ("A","C"), ("B","C"), ("C","D"), ("D","E")])

print("adjacency matrix:\n", nx.to_numpy_array(G, nodelist=["A","B","C","D","E"]))
print("\nadjacency list:", {n: list(G.neighbors(n)) for n in G.nodes})

#### 1. Centrality measures

Degree centrality: just the count of edges touching a node. Worked, on the toy graph: degree(A)=2, degree(B)=2, degree(C)=3, degree(D)=2, degree(E)=1. C has the most connections, the first, cheapest signal for "this account looks like a hub."

Betweenness centrality: how often a node lies on the SHORTEST PATH between every other pair of nodes. A node can have modest degree but high betweenness if it is the only bridge between two otherwise-separate parts of the graph. In this toy graph, D has low degree (2) but every path from E to anywhere else in the graph MUST pass through D, D is a bridge, that structural role is what betweenness captures and degree alone misses entirely.

Closeness centrality: inverse of the average shortest-path distance from a node to every other node, a high closeness node can reach everything else in the graph quickly.

Eigenvector centrality: a node is important if it is connected to OTHER important nodes, not just many nodes, recursive definition (this is the same underlying idea PageRank formalizes for directed graphs, covered below).

In [ ]:
print("degree centrality:", nx.degree_centrality(G))
print("betweenness centrality:", {k: round(v, 3) for k, v in nx.betweenness_centrality(G).items()})
print("closeness centrality:", {k: round(v, 3) for k, v in nx.closeness_centrality(G).items()})
print("eigenvector centrality:", {k: round(v, 3) for k, v in nx.eigenvector_centrality(G).items()})

#### 2. Community detection

Connected components, the simplest version: groups of nodes reachable from each other, with zero edges connecting different groups. In this toy graph everything is one connected component (a path exists between every pair), a genuinely disconnected fraud ring would show up as its own separate component entirely, no edges at all linking it to the legitimate-account graph.

Louvain method, for finding communities WITHIN a single connected graph (not just disconnected components): greedily groups nodes to maximize modularity, a score comparing the density of edges INSIDE proposed communities against what you would expect from a random graph with the same degree distribution. A dense subgroup like {A,B,C} in the toy graph (a full triangle, maximum possible internal density) scores as a strong community, exactly the shape a tight-knit fraud ring sharing multiple payment methods with each other would produce.

In [ ]:
print("connected components:", list(nx.connected_components(G)))

communities = nx.community.louvain_communities(G, seed=42)
print("Louvain communities:", communities)

#### 3. PageRank, worked by hand

Formula: PR(p) = (1-d)/N + d * sum(PR(i)/L(i) for every node i linking TO p), where d=damping factor (0.85 standard), N=total nodes, L(i)=out-degree of node i.

Toy setup, directed graph, 3 nodes: A links to B and A links to C (out-degree 2), B links to C (out-degree 1), C links to A (out-degree 1). Initial PR = 1/3 = 0.333 for all.
```
PR(A) = 0.05 + 0.85 * (PR(C)/1) = 0.05 + 0.85*0.333 = 0.333
PR(B) = 0.05 + 0.85 * (PR(A)/2) = 0.05 + 0.85*0.1667 = 0.192
PR(C) = 0.05 + 0.85 * (PR(A)/2 + PR(B)/1) = 0.05 + 0.85*(0.1667+0.333) = 0.475
```
(check: 0.333+0.192+0.475 = 1.0, correctly normalized)

After just 1 iteration, C already has the highest PageRank (0.475), it receives links from BOTH other nodes. B has the lowest (0.192), only A points to it, and A splits its "vote" between B and C. This is the core PageRank intuition: importance flows through incoming links, and a link from a node that itself only has a few outgoing links carries more weight per link than a link from a node with many outgoing links (dividing by L(i) is exactly this "split the vote" mechanic).

In [ ]:
DG = nx.DiGraph()
DG.add_edges_from([("A","B"), ("A","C"), ("B","C"), ("C","A")])

pagerank = nx.pagerank(DG, alpha=0.85)
print("PageRank (converged):", {k: round(v, 3) for k, v in pagerank.items()})

#### 4. Relevance to fraud detection

This is exactly the technique originally scoped as notebook 4 (`04-graph-fraud-rings.ipynb`) in the fraud-detection-agent project's plan: build a graph where nodes are accounts/transactions and edges are shared identifiers (device fingerprint, phone number, payment method, IP address), then look for unusually dense connected components (a fraud ring sharing infrastructure) or unusually high centrality/PageRank nodes (a hub account, possibly a mule account receiving funds from many other compromised accounts). A flat tabular row-per-transaction model can never see this pattern, the signal lives entirely in the CONNECTIONS between rows, which is exactly what a graph representation, and none of the tabular models in `boosting.ipynb`/`bagging.ipynb`/`classical-ml.ipynb`, is built to surface.